In [39]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from google.colab import drive

drive.mount("/content/gdrive")

%cd "/content/gdrive/MyDrive/datasets"

scaler = joblib.load("scaler2.pkl")
model = tf.keras.models.load_model("best_model2.keras")
model.summary()

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/MyDrive/datasets


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,253 (4.90 KB)

 Trainable params: 417 (1.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 836 (3.27 KB)

In [40]:
EXPECTED_COLUMNS = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time",
    "academic_work_impact",
    "gender_Female",
    "gender_Male",
    "gender_Other",
    "stress_level_High",
    "stress_level_Low",
    "stress_level_Medium"
]

NUMERIC_COLUMNS = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

def preprocesado(input):
  df = input.copy()
  df['academic_work_impact'] = df['academic_work_impact'].map({'Yes': 1, 'No': 0})
  df = pd.get_dummies(df, columns=['gender', 'stress_level'], drop_first=False)
  df = df.reindex(columns=EXPECTED_COLUMNS, fill_value=0)
  df[NUMERIC_COLUMNS] = scaler.transform(df[NUMERIC_COLUMNS])

  return df

In [49]:
prueba = pd.DataFrame([{
     "age": 21,
     "gender": "Male",
     "daily_screen_time_hours": 5.5,
     "social_media_hours": 5.2,
     "gaming_hours": 2.0,
     "work_study_hours": 3.5,
     "sleep_hours": 10.5,
     "notifications_per_day": 14,
     "app_opens_per_day": 14,
     "weekend_screen_time": 2.0,
     "academic_work_impact": "Yes",
     "stress_level": "Low"
 }])

In [50]:
datos = preprocesado(prueba)

prediccion = model.predict(datos)
clase = (prediccion > 0.5).astype(int)

print("Probabilidad:", prediccion[0][0])
if clase[0][0] == 0:
  print("Clase predicha:", "No tiene adicción")
else:
  print("Clase predicha:", "Si tiene adicción")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Probabilidad: 0.99999905
Clase predicha: Si tiene adicción
